In [77]:
%pip install scipy pandas scikit-learn tensorflow keras keras-tuner

import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn import metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from keras import models, Input
from keras import optimizers as opt
from keras import backend as K
from keras.layers import Dense
from keras_tuner.tuners import RandomSearch
from tensorflow.keras.utils import to_categorical
from keras.optimizers import Adam
import importlib
import os
import numpy as np
import pandas as pd
import scipy
import variablesEEGMAT as v # this is where they are getting the test types and data types from


Note: you may need to restart the kernel to use updated packages.


Functions


In [78]:
def get_sorted_mat_files():
    """Return sorted list of .mat files in DIR_RAW."""
    return sorted([f for f in os.listdir(v.DIR_RAW) if f.endswith(".mat")])

def load_dataset(data_type="raw"):
    """
    Loads EEGMAT data.
    Returns:
        dataset: list of arrays, each shape = (21, n_samples)
    """
    if data_type == "raw":
        dir = v.DIR_RAW
    
    dataset = []
    mat_files = get_sorted_mat_files()

    for f in mat_files:
        full_path = os.path.join(dir, f)
        print(f"Loading {f}...")
        
        mat = scipy.io.loadmat(full_path)
        key = [k for k in mat.keys() if not k.startswith("__")][0]
        data = mat[key]   # shape (21, n_samples) but n_samples varies

        print(f, data.shape)
        dataset.append(data)

    # DO NOT convert to np.array → recordings have variable length!
    return dataset


def get_label_from_filename(filename):
    """
    EEGMAT labeling:
    *_1.mat → baseline/rest → label 0
    *_2.mat → mental arithmetic → label 1
    """
    if filename.endswith("_1.mat"):
        return 0
    elif filename.endswith("_2.mat"):
        return 1
    else:
        raise ValueError(f"Unexpected filename format: {filename}")


def load_labels():
    """Load labels for EEGMAT trials."""
    labels = []
    mat_files = get_sorted_mat_files()

    for filename in mat_files:
        labels.append(get_label_from_filename(filename))

    return np.array(labels)


def split_data(dataset, sfreq):
    """
    Split each EEGMAT recording individually into 1-second epochs.
    dataset: list of arrays, each shape (21, n_samples)
    Returns:
        epoched_all: list of arrays, each shape (n_epochs_i, 21, sfreq)
    """
    epoched_all = []

    for data in dataset:
        n_channels, n_samples = data.shape
        n_epochs = n_samples // sfreq

        epoched = np.zeros((n_epochs, n_channels, sfreq))

        for j in range(n_epochs):
            epoched[j] = data[:, j*sfreq:(j+1)*sfreq]

        epoched_all.append(epoched)

    print("Finished epoching all trials.")
    return epoched_all


In [79]:
# from datasetEEGMAT import load_dataset, load_labels, split_data, format_labels
# from featuresEEGMAT import time_series_features, fractal_features, entropy_features, hjorth_features, freq_band_features
import variablesEEGMAT as v
import datasetEEGMAT
import variablesEEGMAT
import featuresEEGMAT  

# reload after making edits to those files
importlib.reload(datasetEEGMAT)
importlib.reload(variablesEEGMAT)
importlib.reload(featuresEEGMAT)

<module 'featuresEEGMAT' from '/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/EEGMAT/featuresEEGMAT.py'>

# Variables

In [80]:
data_type = "raw"
test_type = "task"
data_dir = "data/converted"


# Load Dataset

In [ ]:
dataset_ = load_dataset(data_type=data_type)
label_ = load_labels()

# Step 1: epoch each subject separately
epoched_list = split_data(dataset_, v.SFREQ)
# epoched_list is a LIST of arrays: [(n_epochs_i, 21, 128), ...]

# Step 2: flatten epochs into a single array
X = np.concatenate(epoched_list, axis=0)   # shape = (total_epochs, 21, 128)

# Step 3: expand labels to match epochs
y = np.concatenate([
    np.full(len(e), lbl)
    for e, lbl in zip(epoched_list, label_)
])




Loading Subject00_1.mat...
Subject00_1.mat (21, 91000)
Loading Subject00_2.mat...
Subject00_2.mat (21, 31000)
Loading Subject01_1.mat...
Subject01_1.mat (21, 91000)
Loading Subject01_2.mat...
Subject01_2.mat (21, 31000)
Loading Subject02_1.mat...
Subject02_1.mat (21, 91000)
Loading Subject02_2.mat...
Subject02_2.mat (21, 31000)
Loading Subject03_1.mat...
Subject03_1.mat (21, 91000)
Loading Subject03_2.mat...
Subject03_2.mat (21, 31000)
Loading Subject04_1.mat...
Subject04_1.mat (21, 85000)
Loading Subject04_2.mat...
Subject04_2.mat (21, 31000)
Loading Subject05_1.mat...
Subject05_1.mat (21, 91000)
Loading Subject05_2.mat...
Subject05_2.mat (21, 31000)
Loading Subject06_1.mat...
Subject06_1.mat (21, 91000)
Loading Subject06_2.mat...
Subject06_2.mat (21, 31000)
Loading Subject07_1.mat...
Subject07_1.mat (21, 91000)
Loading Subject07_2.mat...
Subject07_2.mat (21, 31000)
Loading Subject08_1.mat...
Subject08_1.mat (21, 91000)
Loading Subject08_2.mat...
Subject08_2.mat (21, 31000)
Loading Su

/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/mne_features/univariate.py:1055: RuntimeWarning: invalid value encountered in divide
  ln = np.log10(np.divide(ll, a))
/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/mne_features/univariate.py:1058: RuntimeWarning: invalid value encountered in divide
  katz = np.divide(ln, np.add(ln, np.log10(np.divide(d, ll))))


NaNs after fractal features: False


# Compute Features

In [86]:
# features = time_series_features(dataset)
# freq_bands = np.array([1, 4, 8, 12, 30, 50])
# features = freq_band_features(dataset, freq_bands)
# features = hjorth_features(dataset)
# features = entropy_features(dataset)
import numpy as np
import mne_features.univariate as mne_f
def fractal_features(data):
    """
    Computes the Higuchi and Katz fractal dimensions robustly.
    """

    n_trials, n_secs, n_channels, _ = data.shape
    features_per_channel = 2

    features = np.empty((n_trials, n_secs, n_channels * features_per_channel))

    # Normalize and sanitize
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    data = data - np.mean(data, axis=-1, keepdims=True)
    data = data / (np.std(data, axis=-1, keepdims=True) + 1e-8)

    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            # Compute both features, catching numerical warnings
            try:
                higuchi = mne_f.compute_higuchi_fd(second)
            except Exception:
                higuchi = np.zeros(second.shape[0])

            try:
                katz = mne_f.compute_katz_fd(second)
                # Replace inf/nan with zeros or finite values
                katz = np.nan_to_num(katz, nan=0.0, posinf=0.0, neginf=0.0)
            except Exception:
                katz = np.zeros(second.shape[0])

            features[i, j] = np.concatenate([higuchi, katz])

    # Final cleanup
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    print("NaNs after fractal features:", np.isnan(features).any())

    return features.reshape(n_trials * n_secs, n_channels * features_per_channel)

def time_series_features(data):
    '''
    Computes the features variance, RMS and peak-to-peak amplitude using the package mne_features.

    Args:
        data (ndarray): EEG data.

    Returns:
        ndarray: Computed features.

    '''

    n_trials, n_secs, n_channels, _ = data.shape
    features_per_channel = 3

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            variance = mne_f.compute_variance(second)
            rms = mne_f.compute_rms(second)
            ptp_amp = mne_f.compute_ptp_amp(second)
            features[i][j] = np.concatenate([variance, rms, ptp_amp])
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features

def freq_band_features(data, freq_bands):
    '''
    Computes the frequency bands delta, theta, alpha, beta and gamma using the package mne_features.

    Args:
        data (ndarray): EEG data.
        freq_bands (ndarray): The frequency bands to compute.

    Returns:
        ndarray: Computed features.
    '''
    n_trials, n_secs, n_channels, sfreq = data.shape
    features_per_channel = len(freq_bands)-1

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            psd = mne_f.compute_pow_freq_bands(
                sfreq, second, freq_bands=freq_bands)
            features[i][j] = psd
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features
def hjorth_features(data):
    '''
    Computes the features Hjorth mobility (spectral) and Hjorth complexity (spectral) using the package mne_features.

    Args:
        data (ndarray): EEG data.

    Returns:
        ndarray: Computed features.
    '''
    n_trials, n_secs, n_channels, sfreq = data.shape
    features_per_channel = 2

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            mobility_spect = mne_f.compute_hjorth_mobility_spect(sfreq, second)
            complexity_spect = mne_f.compute_hjorth_complexity_spect(
                sfreq, second)
            features[i][j] = np.concatenate([mobility_spect, complexity_spect])
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features

def entropy_features(data):
    '''
    Computes the features Approximate Entropy, Sample Entropy, Spectral Entropy and SVD entropy using the package mne_features.

    Args:
        data (ndarray): EEG data.

    Returns:
        ndarray: Computed features.

    '''
    n_trials, n_secs, n_channels, sfreq = data.shape
    features_per_channel = 4

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            app_entropy = mne_f.compute_app_entropy(second)
            samp_entropy = mne_f.compute_samp_entropy(second)
            spect_entropy = mne_f.compute_spect_entropy(sfreq, second)
            svd_entropy = mne_f.compute_svd_entropy(second)
            features[i][j] = np.concatenate(
                [app_entropy, samp_entropy, spect_entropy, svd_entropy])
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features
# Step 4: NOW compute fractal features (X is not a list, but a 3D array)
features = fractal_features(X.reshape(1, X.shape[0], X.shape[1], X.shape[2]))
# features = time_series_features(dataset)
# freq_bands = np.array([1, 4, 8, 12, 30, 50])
# features = freq_band_features(dataset, freq_bands)
# features = hjorth_features(dataset)
# features = entropy_features(dataset)
data = features
print(len(data))


NaNs after fractal features: False
8676


# k-NN Classifier

In [ ]:
x, x_test, y, y_test = train_test_split(
    data, label, test_size=0.2, random_state=1)
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.25, random_state=1)
scaler = MinMaxScaler()
scaler.fit(x_train)
x = scaler.transform(x)
x_train = scaler.transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

param_grid = {
    'leaf_size': range(50),
    'n_neighbors': range(1, 10),
    'p': [1, 2]
}
split_index = [-1 if x in range(len(x_train)) else 0 for x in range(len(x))]
ps = PredefinedSplit(test_fold=split_index)
knn_clf = GridSearchCV(KNeighborsClassifier(), param_grid, cv=ps, refit=True)
knn_clf.fit(x, y)

KeyboardInterrupt: 

In [ ]:
y_pred = knn_clf.predict(x_test)
y_true = y_test


In [ ]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.50      0.61       139
           1       0.81      0.94      0.87       308

    accuracy                           0.81       447
   macro avg       0.80      0.72      0.74       447
weighted avg       0.80      0.81      0.79       447

[[ 69  70]
 [ 17 291]]


# SVM Classifier

In [87]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, PredefinedSplit

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("svm", SVC())
])

x, x_test, y, y_test = train_test_split(
    data, label, test_size=0.2)
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.25)

param_grid = {
    'svm__C': [0.1, 1, 10, 100, 1000],
    'svm__kernel': ['rbf']
}
split_index = [-1 if x in range(len(x_train)) else 0 for x in range(len(x))]
ps = PredefinedSplit(test_fold=split_index)
svm_clf = GridSearchCV(pipeline, param_grid, cv=ps, refit=True)
svm_clf.fit(x, y)

NameError: name 'label' is not defined

In [ ]:
y_pred = svm_clf.predict(x_test)
y_true = y_test

In [20]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

NameError: name 'y_true' is not defined

In [ ]:
loaded_svm_model = joblib.load("svm_model.pkl")
pred = loaded_svm_model.predict(x_test)

# Multilayer Perceptron

In [ ]:
K.clear_session()
y_v = label
y_v = to_categorical(y_v)
x_train, x_test, y_train, y_test = train_test_split(
    data, y_v, test_size=0.2, random_state=1)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.25, random_state=1)

In [ ]:
import keras

def model_builder(hp):
    model = models.Sequential()
    model.add(Input(shape=(x_train.shape[1],)))

    for i in range(hp.Int('layers', 2, 6)):
        model.add(Dense(units=hp.Int('units_' + str(i), 32, 1024, step=32),
                        activation=hp.Choice('act_' + str(i), ['relu', 'sigmoid'])))

    model.add(Dense(v.N_CLASSES, activation='softmax', name='out'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss="binary_crossentropy",
                  metrics=['accuracy'])
    return model

In [ ]:
tuner = RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=15,
    executions_per_trial=2,
    overwrite=True
)

In [ ]:
tuner.search(x_train, y_train, epochs=50, validation_data=[x_val, y_val])

Trial 15 Complete [00h 00m 39s]
val_accuracy: 0.7337807416915894

Best val_accuracy So Far: 0.7393735945224762
Total elapsed time: 00h 08m 16s


In [ ]:
model = tuner.get_best_models(num_models=1)[0]

/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
y_pred = model.predict(x_test)
y_true = y_test
y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_true, axis=1)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [ ]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.24      0.36       139
           1       0.74      0.95      0.83       308

    accuracy                           0.73       447
   macro avg       0.71      0.60      0.59       447
weighted avg       0.72      0.73      0.68       447

[[ 34 105]
 [ 16 292]]


In [92]:
import joblib
loaded_model = joblib.load("svm_model.pkl")

def pad_features_to_32_channels(features):
    """
    Pads each feature vector from 21 channels → 32 channels.
    features shape: (n_samples, 42)   # 21 ch * 2 fracs
    returns shape: (n_samples, 64)    # 32 ch * 2 fracs
    """
    n_samples = features.shape[0]
    padded = np.zeros((n_samples, 32*2))  # 64 dims
    padded[:, :features.shape[1]] = features   # fill first 42 dims
    return padded


data = pad_features_to_32_channels(data)
pred = loaded_model.predict(data)

print(metrics.classification_report(y, pred))
print(metrics.confusion_matrix(y, pred))


              precision    recall  f1-score   support

           0       0.74      1.00      0.85      6444
           1       0.00      0.00      0.00      2232

    accuracy                           0.74      8676
   macro avg       0.37      0.50      0.43      8676
weighted avg       0.55      0.74      0.63      8676

[[6444    0]
 [2232    0]]


/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 